# Brain Tumor Segmentation — SE-UNet++ — **Seed 2** (frozen split, per-patient) — Kaggle

This notebook trains/evaluates **only seed 2**. The 5 notebooks (seed 1–5) are identical except for the seed, and they all use the **same frozen patient split** (split RNG is fixed by `SEED = 42`, independent of the training seed).

**Kaggle setup:**
1. Session options → **Accelerator: GPU T4 x2** (or P100).
2. **Add Input** → `dschettler8845/brats-2021-task1`.
3. Optional: `QUICK_TEST = True` in the runner cell for a 3-epoch timing check, then set it back to `False`.
4. Use **Save Version → Save & Run All (Commit)** for the full run.

Outputs (`/kaggle/working`): `SE-UNet++_summary.csv`, `SE-UNet++_per_patient.csv`, `SE-UNet++_seed2_best.pt`, `SE-UNet++_seed2_curve.png`, `SE-UNet++_seed2_epochlog.csv`, `split_patients.json`.

In [ ]:
import os, glob, random, json, csv, time
T_START = time.time()
import numpy as np
import pandas as pd
try:
    import nibabel as nib
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nibabel"])
    import nibabel as nib
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
try:
    from torchinfo import summary as torchinfo_summary
    HAS_TORCHINFO = True
except Exception:
    HAS_TORCHINFO = False

# ---- paths (Kaggle) ----
DATA_ROOTS = ["/kaggle/input"]
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./brats_out"
# slice cache is big (~7 GB) -> keep it OUT of /kaggle/working (20 GB limit, and it would be saved as output)
_tmp = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
PROCESSED_DIR = os.path.join(_tmp, "brats_processed")

# ---- data ----
NUM_PATIENTS = 500
VAL_FRAC, TEST_FRAC = 0.20, 0.20
N_VAL_PATIENTS = round(NUM_PATIENTS * VAL_FRAC)
N_TEST_PATIENTS = round(NUM_PATIENTS * TEST_FRAC)
IMG_SIZE = 128
NEIGHBOR_SLICES = 2
KEEP_EMPTY_FRACTION = 0.10
MAX_SLICES_PER_PATIENT = 100
MIN_BRAIN_FRACTION = 0.02
STORE_FLOAT16 = True
FORCE_PREPROCESS = False

# ---- model / training (same recipe as the original notebook) ----
BASE_FILTERS = 64
IN_CHANNELS = 4              # FLAIR, T1, T1ce, T2
OUT_CHANNELS = 3             # WT, TC, ET
EPOCHS = 100
PATIENCE = 10                # early stopping
BATCH_SIZE = 8               # kept at 8 so results are comparable with the other models
LR = 1e-4
WEIGHT_DECAY = 1e-4
RLROP_FACTOR = 0.5
RLROP_PATIENCE = 4
MIN_LR = 1e-7
NUM_WORKERS = 0
SEED = 42

# ---- Kaggle session guard (GPU session limit = 12 h) ----
TIME_BUDGET_H = 11.0

MODALITIES = ["flair", "t1", "t1ce", "t2"]
REGIONS = ["WT", "TC", "ET"]
DISK_DTYPE = np.float16 if STORE_FLOAT16 else np.float32

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
torch.backends.cudnn.benchmark = True

print("torch", torch.__version__)
print(f"Patients {NUM_PATIENTS}: {NUM_PATIENTS-N_VAL_PATIENTS-N_TEST_PATIENTS} train / {N_VAL_PATIENTS} val / {N_TEST_PATIENTS} test")
print(f"img={IMG_SIZE} | batch={BATCH_SIZE} | base_filters={BASE_FILTERS} | epochs={EPOCHS} (patience {PATIENCE}) | LR={LR}")
print("Cache:", PROCESSED_DIR, "| Outputs:", OUT_DIR)

In [ ]:
def get_device():
    if not torch.cuda.is_available():
        raise SystemExit("No GPU. Session options -> Accelerator -> GPU T4 / P100, then restart.")
    print("GPU:", torch.cuda.get_device_name(0), "(sm_%d%d)" % torch.cuda.get_device_capability(0),
          "| VRAM %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
    _ = torch.randn(1024, 1024, device="cuda") @ torch.randn(1024, 1024, device="cuda")
    torch.cuda.synchronize(); print("GPU compute test: OK")
    return torch.device("cuda")

device = get_device()

## 1. Find BraTS patients (reads directly from the dataset's .tar files)

In [ ]:
import tarfile, gzip, re
# Works with the Kaggle dataset dschettler8845/brats-2021-task1 (patients live inside .tar files)
# and also with already-extracted folders. Nothing is extracted to disk: volumes are read straight from the tar.
SRC = {}       # pid -> {"flair"/"t1"/"t1ce"/"t2"/"seg": path-or-(tarpath, TarInfo)}
_TARS = {}     # open tar handles
_MOD_RE = re.compile(r"(?:^|/)([^/]+)_(flair|t1ce|t1|t2|seg)\.nii(?:\.gz)?$")

def index_patients(roots):
    # 1) already-extracted folders
    for root in roots:
        if not os.path.isdir(root): continue
        for f in glob.glob(os.path.join(root, "**", "*.nii*"), recursive=True):
            m = _MOD_RE.search(f.replace("\\", "/"))
            if m: SRC.setdefault(m.group(1), {})[m.group(2)] = f
    # 2) .tar archives (read headers only: fast)
    if not any(all(k in v for k in MODALITIES + ["seg"]) for v in SRC.values()):
        tars = sorted({t for root in roots if os.path.isdir(root) for t in glob.glob(os.path.join(root, "**", "*.tar"), recursive=True)})
        print("tar files:", [os.path.basename(t) for t in tars])
        for t in tars:
            try:
                tf = tarfile.open(t); t0 = time.time()
                for mem in tf:                       # reads headers only (tar is uncompressed)
                    m = _MOD_RE.search(mem.name)
                    if mem.isfile() and m: SRC.setdefault(m.group(1), {})[m.group(2)] = (t, mem)
                _TARS[t] = tf; print(f"  indexed {os.path.basename(t)} in {time.time()-t0:.0f}s")
            except tarfile.ReadError as e:
                print("  skip", t, e)
    pids = sorted(p for p, v in SRC.items() if all(k in v for k in MODALITIES + ["seg"]))   # need 4 modalities + seg (drops the validation set)
    if not pids:
        raise SystemExit("No patients found. Add the dataset 'dschettler8845/brats-2021-task1' via 'Add Input'.")
    print(f"Found {len(pids)} patients with all 4 modalities + seg")
    return pids

def load_vol(pid, m):
    s = SRC[pid][m]
    if isinstance(s, str):
        return nib.load(s).get_fdata().astype(np.float32)
    tpath, mem = s
    data = _TARS[tpath].extractfile(mem).read()
    if mem.name.endswith(".gz"): data = gzip.decompress(data)
    return nib.Nifti1Image.from_bytes(data).get_fdata().astype(np.float32)

patients = index_patients(DATA_ROOTS)      # sorted patient IDs (same order as the local sorted folders -> same frozen split)
print("Example:", patients[0], "| Last:", patients[-1])
assert len(patients) >= NUM_PATIENTS, f"Only {len(patients)} patients found, NUM_PATIENTS={NUM_PATIENTS}"
_t = load_vol(patients[0], "flair"); print("test read flair:", _t.shape, _t.dtype)

## 2. Preprocessing → on-disk slice cache (runs once per session)

In [ ]:
def znorm_channel(vol):
    brain = vol > 0
    if brain.sum() == 0: return vol
    vol = (vol - vol[brain].mean()) / (vol[brain].std() + 1e-8); vol[~brain] = 0.0
    return vol

def rz_img(sl): return cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
def rz_msk(sl): return cv2.resize(sl.astype(np.uint8), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

def select_slices(seg, mods, Z):
    brain_ok = [(mods[0][:, :, z] != 0).mean() >= MIN_BRAIN_FRACTION for z in range(Z)]
    tumor = [z for z in range(Z) if brain_ok[z] and seg[:, :, z].max() > 0]
    ctx = set()
    for tz in tumor:
        for dz in range(-NEIGHBOR_SLICES, NEIGHBOR_SLICES + 1):
            zz = tz + dz
            if 0 <= zz < Z and brain_ok[zz]: ctx.add(zz)
    ctx -= set(tumor)
    empties = [z for z in range(Z) if brain_ok[z] and z not in set(tumor) | ctx]
    random.shuffle(empties); empties = empties[:int(len(empties) * KEEP_EMPTY_FRACTION)]
    keep = list(tumor) + sorted(ctx) + empties
    if MAX_SLICES_PER_PATIENT and len(keep) > MAX_SLICES_PER_PATIENT: keep = keep[:MAX_SLICES_PER_PATIENT]
    return sorted(set(keep))

img_dir = os.path.join(PROCESSED_DIR, "images"); msk_dir = os.path.join(PROCESSED_DIR, "masks")

def preprocess_to_disk(patient_dirs):
    meta_path = os.path.join(PROCESSED_DIR, "meta.json")
    if (not FORCE_PREPROCESS) and os.path.exists(meta_path):
        meta = json.load(open(meta_path))
        if meta.get("num_patients") == NUM_PATIENTS and meta.get("img_size") == IMG_SIZE:
            print("Found existing cache -> skipping preprocessing."); return meta["splits"]
    os.makedirs(img_dir, exist_ok=True); os.makedirs(msk_dir, exist_ok=True)
    pdirs = list(patient_dirs); random.shuffle(pdirs); pdirs = pdirs[:NUM_PATIENTS]
    val_set = set(os.path.basename(p) for p in pdirs[:N_VAL_PATIENTS])
    test_set = set(os.path.basename(p) for p in pdirs[N_VAL_PATIENTS:N_VAL_PATIENTS + N_TEST_PATIENTS])
    splits = {"train": [], "val": [], "test": []}
    for pdir in tqdm(pdirs, desc="Preprocessing to disk"):
        pid = os.path.basename(pdir)
        mods = [znorm_channel(load_vol(pdir, m)) for m in MODALITIES]
        seg = load_vol(pdir, "seg"); Z = seg.shape[2]
        which = "val" if pid in val_set else "test" if pid in test_set else "train"
        for z in select_slices(seg, mods, Z):
            sid = f"{pid}_z{z:03d}"
            img = np.stack([rz_img(m[:, :, z]) for m in mods], 0).astype(DISK_DTYPE)
            s = seg[:, :, z]
            mask = np.stack([rz_msk(np.isin(s, [1, 2, 4])), rz_msk(np.isin(s, [1, 4])), rz_msk(s == 4)], 0).astype(np.uint8)
            np.save(os.path.join(img_dir, sid + ".npy"), img); np.save(os.path.join(msk_dir, sid + ".npy"), mask)
            splits[which].append(sid)
    json.dump({"num_patients": NUM_PATIENTS, "img_size": IMG_SIZE, "splits": splits}, open(meta_path, "w"))
    return splits

splits = preprocess_to_disk(patients)
# save patient-level split to outputs -> lets you verify it matches the split of your other models
_split_pat = {k: sorted({s.rsplit("_z", 1)[0] for s in v}) for k, v in splits.items()}
json.dump(_split_pat, open(os.path.join(OUT_DIR, "split_patients.json"), "w"), indent=1)

n = sum(len(v) for v in splits.values())
per = 4 * IMG_SIZE * IMG_SIZE * (2 if STORE_FLOAT16 else 4) + 3 * IMG_SIZE * IMG_SIZE
for k in ["train", "val", "test"]:
    print(f"{k.upper():5s}: {len(_split_pat[k]):4d} patients | {len(splits[k]):6d} slices")
print(f"Total {n} slices ~ {n*per/1e9:.1f} GB at {PROCESSED_DIR}")

## 3. Datasets + DataLoaders (RAM cache; augmentation runs on the GPU in the runner)

In [ ]:
CACHE_IN_RAM = True
_RAM_CACHE = {}

class DiskSliceDS(Dataset):
    def __init__(self, ids, aug=False): self.ids, self.aug = ids, aug
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        sid = self.ids[i]
        if CACHE_IN_RAM:
            rec = _RAM_CACHE.get(sid)
            if rec is None:
                xr = np.load(os.path.join(img_dir, sid + ".npy"))
                yr = np.load(os.path.join(msk_dir, sid + ".npy"))
                _RAM_CACHE[sid] = (xr, yr)
            else:
                xr, yr = rec
            x = xr.astype(np.float32); y = yr.astype(np.float32)
        else:
            x = np.load(os.path.join(img_dir, sid + ".npy")).astype(np.float32)
            y = np.load(os.path.join(msk_dir, sid + ".npy")).astype(np.float32)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y))

train_loader = DataLoader(DiskSliceDS(splits["train"], False), BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(DiskSliceDS(splits["val"], False),   BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(DiskSliceDS(splits["test"], False),  BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
xb, yb = next(iter(train_loader)); print("Batch images:", tuple(xb.shape), "| masks:", tuple(yb.shape))

## 4. U-Net++ with SE decoder

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__(); self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(True))
    def forward(self, x): return self.net(x)

class SEBlock(nn.Module):
    """Squeeze-and-Excitation: recalibrate channels (used in the decoder)."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(channels, hidden, bias=False), nn.ReLU(inplace=True),
                                nn.Linear(hidden, channels, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c, _, _ = x.shape
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)

class SEConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__(); self.conv = ConvBlock(cin, cout); self.se = SEBlock(cout)
    def forward(self, x): return self.se(self.conv(x))

class UNetPP_SE(nn.Module):
    def __init__(self, in_ch=4, out_ch=3, base=64):
        super().__init__(); self.pool = nn.MaxPool2d(2)
        f = [base, base * 2, base * 4, base * 8, base * 16]
        self.x00 = ConvBlock(in_ch, f[0]); self.x10 = ConvBlock(f[0], f[1]); self.x20 = ConvBlock(f[1], f[2])
        self.x30 = ConvBlock(f[2], f[3]); self.x40 = ConvBlock(f[3], f[4])
        self.dec = nn.ModuleDict({f"x{i}{j}": SEConvBlock(j * f[i] + f[i + 1], f[i])
                                  for j in range(1, 5) for i in range(0, 5 - j)})
        self.out = nn.Conv2d(f[0], out_ch, 1)
    def _up(self, x, ref):
        return F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)
    def forward(self, x):
        X = {}
        X[(0, 0)] = self.x00(x)
        X[(1, 0)] = self.x10(self.pool(X[(0, 0)]))
        X[(2, 0)] = self.x20(self.pool(X[(1, 0)]))
        X[(3, 0)] = self.x30(self.pool(X[(2, 0)]))
        X[(4, 0)] = self.x40(self.pool(X[(3, 0)]))
        for j in range(1, 5):
            for i in range(0, 5 - j):
                feats = [X[(i, k)] for k in range(j)] + [self._up(X[(i + 1, j - 1)], X[(i, 0)])]
                X[(i, j)] = self.dec[f"x{i}{j}"](torch.cat(feats, 1))
        return self.out(X[(0, 4)])

_m = UNetPP_SE(IN_CHANNELS, OUT_CHANNELS, BASE_FILTERS)
print(f"SE-UNet++ parameters: {sum(p.numel() for p in _m.parameters())/1e6:.2f}M"); del _m

## 5. Loss (Dice only) and metrics

In [ ]:
class DiceLoss(nn.Module):
    def forward(self, logits, y, eps=1e-6):
        p = torch.sigmoid(logits)
        inter = (p * y).sum(); denom = p.sum() + y.sum()
        return 1 - (2 * inter + eps) / (denom + eps)

@torch.no_grad()
def accumulate(logits, y, acc, thr=0.5):
    p = (torch.sigmoid(logits) > thr).float()
    acc["tp"] += (p * y).sum((0, 2, 3)).cpu()
    acc["fp"] += (p * (1 - y)).sum((0, 2, 3)).cpu()
    acc["fn"] += ((1 - p) * y).sum((0, 2, 3)).cpu()
    return acc

def new_acc(): return {"tp": torch.zeros(3), "fp": torch.zeros(3), "fn": torch.zeros(3)}

def metrics_from(acc, eps=1e-6):
    tp, fp, fn = acc["tp"], acc["fp"], acc["fn"]
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    prec = (tp + eps) / (tp + fp + eps)
    rec = (tp + eps) / (tp + fn + eps)
    return dice, iou, prec, rec

## 6. Train + evaluate seed 2
Every epoch is printed live with batch heartbeats, and appended to `SE-UNet++_seed2_epochlog.csv`. Time guard stops training at `TIME_BUDGET_H` hours.

In [ ]:
import gc
MODEL_NAME   = "SE-UNet++"
SEEDS        = [2]
QUICK_TEST   = False      # True -> 1 seed, 3 epochs (timing check; writes to separate *_QUICK files)
COMPUTE_HD95 = True
MAX_EPOCHS   = 3 if QUICK_TEST else EPOCHS
TAG = "_QUICK" if QUICK_TEST else ""

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
assert len(train_loader.dataset) > 0 and len(splits["test"]) > 0, "Empty split."

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def make_model():
    return UNetPP_SE(IN_CHANNELS, OUT_CHANNELS, BASE_FILTERS)

def _cxpad(t, H, W):
    _, _, h, w = t.shape
    if h > H: top = (h - H) // 2; t = t[:, :, top:top + H, :]
    if w > W: left = (w - W) // 2; t = t[:, :, :, left:left + W]
    _, _, h, w = t.shape; ph = H - h; pw = W - w
    if ph > 0 or pw > 0: t = F.pad(t, [pw // 2, pw - pw // 2, ph // 2, ph - ph // 2])
    return t

def gpu_aug(x, y):
    if random.random() < 0.5: x = torch.flip(x, [3]); y = torch.flip(y, [3])
    if random.random() < 0.5: x = torch.flip(x, [2]); y = torch.flip(y, [2])
    k = random.randint(0, 3)
    if k: x = torch.rot90(x, k, [2, 3]).contiguous(); y = torch.rot90(y, k, [2, 3]).contiguous()
    if random.random() < 0.5:
        s = random.uniform(0.85, 1.15); H, W = x.shape[-2:]
        nh, nw = max(8, int(round(H * s))), max(8, int(round(W * s)))
        x = _cxpad(F.interpolate(x, size=(nh, nw), mode="bilinear", align_corners=False), H, W)
        y = _cxpad(F.interpolate(y, size=(nh, nw), mode="nearest"), H, W)
    if random.random() < 0.5:
        x = x * random.uniform(0.9, 1.1) + random.uniform(-0.1, 0.1)
    return x.contiguous(memory_format=torch.channels_last), y.contiguous()

def hd95_2d(pred, gt):
    from scipy.ndimage import binary_erosion, distance_transform_edt
    pred = pred.astype(bool); gt = gt.astype(bool)
    if pred.sum() == 0 or gt.sum() == 0: return None
    sp = pred ^ binary_erosion(pred); sg = gt ^ binary_erosion(gt)
    if sp.sum() == 0 or sg.sum() == 0: return None
    dg = distance_transform_edt(~sg); dp = distance_transform_edt(~sp)
    return float(np.percentile(np.concatenate([dg[sp], dp[sg]]), 95))

test_by_patient = {}
for sid in splits["test"]:
    test_by_patient.setdefault(sid.rsplit("_z", 1)[0], []).append(sid)
print(f"Test: {len(test_by_patient)} patients, {len(splits['test'])} slices")

@torch.no_grad()
def eval_per_patient(model):
    model.eval(); rows = []; acc = new_acc(); EVAL_BS = min(BATCH_SIZE, 8)
    for _pi, (pid, sids) in enumerate(test_by_patient.items()):
        if _pi % 20 == 0: print(f"    eval patient {_pi+1}/{len(test_by_patient)}", flush=True)
        tp = np.zeros(3); fp = np.zeros(3); fn = np.zeros(3); hd = [[], [], []]
        for k in range(0, len(sids), EVAL_BS):
            chunk = sids[k:k + EVAL_BS]
            x = torch.from_numpy(np.stack([np.load(os.path.join(img_dir, s + ".npy")).astype(np.float32) for s in chunk]))
            y = np.stack([np.load(os.path.join(msk_dir, s + ".npy")).astype(np.uint8) for s in chunk])
            x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            with torch.amp.autocast("cuda"):
                logits = model(x)
            acc = accumulate(logits.float(), torch.from_numpy(y.astype(np.float32)).to(device), acc)
            p = (torch.sigmoid(logits.float()) > 0.5).cpu().numpy().astype(np.uint8)
            for b in range(p.shape[0]):
                for r in range(3):
                    pr, gt = p[b, r], y[b, r]
                    tp[r] += np.logical_and(pr, gt).sum(); fp[r] += np.logical_and(pr, 1 - gt).sum(); fn[r] += np.logical_and(1 - pr, gt).sum()
                    if COMPUTE_HD95:
                        h = hd95_2d(pr, gt)
                        if h is not None: hd[r].append(h)
            del x, logits, p
        eps = 1e-6; dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
        hdm = [(float(np.mean(v)) if len(v) else float("nan")) for v in hd]
        rows.append({"model": MODEL_NAME, "patient_id": pid,
                     "WT_Dice": round(float(dice[0]), 5), "TC_Dice": round(float(dice[1]), 5), "ET_Dice": round(float(dice[2]), 5),
                     "WT_HD95": (round(hdm[0], 3) if hdm[0] == hdm[0] else "NA"),
                     "TC_HD95": (round(hdm[1], 3) if hdm[1] == hdm[1] else "NA"),
                     "ET_HD95": (round(hdm[2], 3) if hdm[2] == hdm[2] else "NA")})
    d, i, p_, r_ = metrics_from(acc)
    return rows, d.tolist(), i.tolist(), p_.tolist(), r_.tolist()

def train_one_seed(seed):
    set_seed(seed)
    model = make_model().to(device).to(memory_format=torch.channels_last)
    loss_fn = DiceLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=RLROP_FACTOR, patience=RLROP_PATIENCE, min_lr=MIN_LR)
    scaler = torch.amp.GradScaler("cuda")
    ckpt = os.path.join(OUT_DIR, f"{MODEL_NAME}_seed{seed}_best.pt")
    _lp0 = os.path.join(OUT_DIR, f"{MODEL_NAME}_seed{seed}_epochlog{TAG}.csv")
    if os.path.exists(_lp0): os.remove(_lp0)
    htr, hva = [], []; best = -1; best_ep = 0; noimp = 0; t0 = time.time(); stop_reason = "max_epochs"
    for ep in range(1, MAX_EPOCHS + 1):
        model.train(); run = 0.0; te = time.time()
        nb_tr = len(train_loader); hb = max(1, nb_tr // 5)
        print(f"[seed {seed}] ep {ep:03d}/{MAX_EPOCHS} started ({nb_tr} batches)", flush=True)
        for bi, (x, y) in enumerate(train_loader):
            if (bi + 1) % hb == 0: print(f"    batch {bi+1}/{nb_tr} | {time.time()-te:.0f}s", flush=True)
            x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last); y = y.to(device, non_blocking=True)
            x, y = gpu_aug(x, y)
            with torch.amp.autocast("cuda"):
                loss = loss_fn(model(x), y)
            if not torch.isfinite(loss): print("non-finite loss, skipping batch", flush=True); continue
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item() * x.size(0)
        tr = run / len(train_loader.dataset)
        model.eval(); run = 0.0; acc = new_acc()
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last); y = y.to(device, non_blocking=True)
                with torch.amp.autocast("cuda"):
                    logits = model(x); loss = loss_fn(logits, y)
                run += loss.item() * x.size(0); acc = accumulate(logits.float(), y, acc)
        va = run / len(val_loader.dataset); dice, _, _, _ = metrics_from(acc); md = dice.mean().item()
        sched.step(va); htr.append(tr); hva.append(va)
        if md > best: best = md; best_ep = ep; noimp = 0; torch.save({"model": model.state_dict()}, ckpt)
        else: noimp += 1
        print(f"[seed {seed}] ep {ep:03d}/{MAX_EPOCHS} | {time.time()-te:5.0f}s | train {tr:.4f} | val {va:.4f} | valDice {md:.3f} | "
              f"elapsed {(time.time()-T_START)/3600:.2f}h" + ("  <- best" if noimp == 0 else f"  (no-improve {noimp}/{PATIENCE})"), flush=True)
        _lp = os.path.join(OUT_DIR, f"{MODEL_NAME}_seed{seed}_epochlog{TAG}.csv")
        with open(_lp, "a", newline="") as _f:
            _w = csv.writer(_f)
            if ep == 1: _w.writerow(["epoch", "sec", "train_loss", "val_loss", "val_mean_dice", "best"])
            _w.writerow([ep, round(time.time()-te, 1), round(tr, 5), round(va, 5), round(md, 5), int(noimp == 0)])
        if noimp >= PATIENCE: stop_reason = "early_stop"; print(f"[seed {seed}] early stop @ {ep} (best {best_ep})", flush=True); break
        if (time.time() - T_START) > TIME_BUDGET_H * 3600:
            stop_reason = "time_budget"; print(f"[seed {seed}] TIME BUDGET reached -> stopping this seed @ {ep}"); break
    ttime = time.time() - t0; tot = len(htr)
    plt.figure(figsize=(6, 3.5)); plt.plot(range(1, tot + 1), htr, label="train"); plt.plot(range(1, tot + 1), hva, label="val")
    plt.title(f"{MODEL_NAME} seed {seed}"); plt.xlabel("epoch"); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{MODEL_NAME}_seed{seed}_curve.png"), dpi=110); plt.close()
    model.load_state_dict(torch.load(ckpt, map_location=device)["model"]); model.eval()
    return model, best, best_ep, tot, ttime, stop_reason

pp_csv  = os.path.join(OUT_DIR, f"{MODEL_NAME}_per_patient{TAG}.csv")
sum_csv = os.path.join(OUT_DIR, f"{MODEL_NAME}_summary{TAG}.csv")
n_params_M = sum(p.numel() for p in make_model().parameters()) / 1e6


def run_seed(seed):
    done = set(int(x) for x in pd.read_csv(sum_csv)["seed"].tolist()) if os.path.exists(sum_csv) else set()
    if seed in done:
        print(f"Seed {seed} already finished (in {os.path.basename(sum_csv)}) -> skipped"); return
    if (time.time() - T_START) > TIME_BUDGET_H * 3600:
        print(f"Time budget used up -> not starting seed {seed}. Re-run it in a new session."); return
    print(f"\n================= {MODEL_NAME} | seed {seed} =================", flush=True)
    model, best_val, best_ep, tot, ttime, why = train_one_seed(seed)
    rows, dice, iou, prec, rec = eval_per_patient(model)
    for r in rows: r["seed"] = seed
    cols = ["model", "seed", "patient_id", "WT_Dice", "TC_Dice", "ET_Dice", "WT_HD95", "TC_HD95", "ET_HD95"]
    wh = not os.path.exists(pp_csv)
    with open(pp_csv, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        if wh: w.writeheader()
        for r in rows: w.writerow({k: r[k] for k in cols})
    def _mhd(idx):
        key = ["WT_HD95", "TC_HD95", "ET_HD95"][idx]; v = [r[key] for r in rows if r[key] != "NA"]
        return round(float(np.mean(v)), 3) if v else "NA"
    srow = {"model": MODEL_NAME, "seed": seed, "params_M": round(n_params_M, 3), "convergence_epoch": best_ep, "total_epochs": tot,
            "stopped_by": why, "train_time_s": round(ttime, 1), "best_val_dice": round(best_val, 5),
            "WT_Dice": round(dice[0], 4), "TC_Dice": round(dice[1], 4), "ET_Dice": round(dice[2], 4), "Mean_Dice": round(float(np.mean(dice)), 4),
            "WT_IoU": round(iou[0], 4), "TC_IoU": round(iou[1], 4), "ET_IoU": round(iou[2], 4), "Mean_IoU": round(float(np.mean(iou)), 4),
            "Precision": round(float(np.mean(prec)), 4), "Recall": round(float(np.mean(rec)), 4),
            "WT_HD95": _mhd(0), "TC_HD95": _mhd(1), "ET_HD95": _mhd(2)}
    wh = not os.path.exists(sum_csv)
    with open(sum_csv, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(srow.keys()))
        if wh: w.writeheader()
        w.writerow(srow)
    print(f"[seed {seed}] test mean Dice={srow['Mean_Dice']}  ({why})  ->  {os.path.basename(pp_csv)}", flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"[seed {seed}] FINISHED | elapsed {(time.time()-T_START)/3600:.2f}h", flush=True)



### Run seed 2

In [ ]:
run_seed(2)

### Result

In [ ]:
print(f"\n========== {MODEL_NAME} | elapsed {(time.time()-T_START)/3600:.2f} h ==========")
display(pd.read_csv(sum_csv).T)